In [ ]:
import glob, os, subprocess, sys, time
WORK = "/kaggle/working"
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "underthesea", "gensim"], check=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)


In [ ]:
from pathlib import Path
FILES = {'src/absa/__init__.py': '"""Vietnamese student feedback ABSA: data, preprocessing, and metrics."""\n', 'src/absa/aspects.py': '"""Aspect descriptions used to initialize E4 queries.\n\nTaken from the annotation guideline (Table 3 of the draft), without its repeated\n"Các ý kiến đề cập đến" prefix, which would pull all queries toward the same vector.\nLocked before error diagnosis: do not edit them based on validation or test errors.\n"""\n\nfrom .data import ASPECTS\n\nASPECT_DESCRIPTIONS = {\n    "ky_nang_giang_day": "Kỹ năng giảng dạy: cách tổ chức dạy học, phương pháp dạy học lý thuyết và thực hành của giảng viên.",\n    "hanh_vi": "Hành vi: hành vi, thái độ của giảng viên trong giảng dạy và giao tiếp với người học.",\n    "de_xuat": "Đề xuất: đề xuất, mong muốn của sinh viên gửi đến giảng viên và nhà trường.",\n    "bai_tap": "Bài tập: bài tập, số lượng bài tập và các loại bài tập của giảng viên.",\n    "chuong_trinh_hoc": "Chương trình học: chương trình học và môn học.",\n    "kien_thuc": "Kiến thức: mức độ hiểu biết của giảng viên về nội dung giảng dạy và kiến thức đã cung cấp cho sinh viên.",\n    "kinh_nghiem": "Kinh nghiệm: kinh nghiệm thực tiễn và việc đưa nội dung thực tiễn lồng ghép vào bài giảng của giảng viên.",\n    "cung_cap_tai_lieu": "Cung cấp tài liệu: việc cung cấp tài liệu, giáo trình, giáo án của giảng viên.",\n    "thiet_bi_day_hoc": "Thiết bị dạy học: trang thiết bị dạy học như phòng học, máy chiếu, quạt và đèn.",\n    "cham_diem": "Chấm điểm: hoạt động chấm điểm của giảng viên.",\n    "noi_chung": "Nói chung: vấn đề chung của giảng viên hoặc những nhận xét không thuộc các khía cạnh khác.",\n}\n\n\ndef aspect_texts() -> list[str]:\n    """Descriptions in the same order as `ASPECTS`."""\n    return [ASPECT_DESCRIPTIONS[name] for name in ASPECTS]\n', 'src/absa/audit.py': '"""Annotation audit (RQ1): draw the blind sample, then score the two filled sheets.\n\n    python -m absa.audit sample\n    python -m absa.audit score\n\nAnnotators receive only `audit_annotator_A.csv` or `audit_annotator_B.csv`. In each aspect\ncolumn they write `pos`, `neg` or `neu` if the sentence mentions that aspect, and leave it\nblank otherwise. `audit_key.csv` holds the current labels and must not be shown to them.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .aspects import ASPECT_DESCRIPTIONS\nfrom .data import ASPECTS, IGNORE_INDEX, POLARITIES, load_split\n\nCODE_TO_POLARITY = {"neg": 0, "neu": 1, "pos": 2}\nPOLARITY_TO_CODE = {POLARITIES.index(name): code for code, name in (("neg", "negative"), ("neu", "neutral"), ("pos", "positive"))}\nSAMPLE_SEED = 20261101\n\n\ndef draw_sample(samples, size: int = 300, per_aspect: int = 30, seed: int = SAMPLE_SEED, exclude: set[str] = frozenset()):\n    """Stratified sample: rarest aspects first until each has `per_aspect`, then uniform fill."""\n    rng = random.Random(seed)\n    pool = [s for s in samples if s.uid not in exclude]\n    rng.shuffle(pool)\n    chosen, chosen_ids = [], set()\n    counts = {a: 0 for a in range(len(ASPECTS))}\n    available = {a: sum(s.aspect[a] > 0 for s in pool) for a in counts}\n    if min(available.values()) < per_aspect:\n        raise ValueError(f"not enough sentences per aspect: {available}")\n\n    for a in sorted(counts, key=available.get):\n        for s in pool:\n            if counts[a] >= per_aspect:\n                break\n            if s.uid not in chosen_ids and s.aspect[a] > 0:\n                chosen.append(s)\n                chosen_ids.add(s.uid)\n                for b in counts:\n                    counts[b] += int(s.aspect[b] > 0)\n    if len(chosen) > size:\n        raise ValueError(f"{len(chosen)} sentences are needed for the strata; raise size or lower per_aspect")\n    stratified = len(chosen)\n    for s in pool:\n        if len(chosen) >= size:\n            break\n        if s.uid not in chosen_ids:\n            chosen.append(s)\n            chosen_ids.add(s.uid)\n    return chosen, stratified\n\n\ndef _write(path: Path, rows, header) -> None:\n    with path.open("w", encoding="utf-8-sig", newline="") as f:\n        writer = csv.writer(f)\n        writer.writerow(header)\n        writer.writerows(rows)\n\n\ndef write_sheets(samples, out_dir: Path, stratified: int, seed: int = SAMPLE_SEED) -> None:\n    """One sheet per annotator with different orders, and one key file for scoring."""\n    out_dir.mkdir(parents=True, exist_ok=True)\n    items = [(f"Q{i:03d}", s) for i, s in enumerate(samples, start=1)]\n    for name, order_seed in (("A", seed + 1), ("B", seed + 2)):\n        shuffled = items[:]\n        random.Random(order_seed).shuffle(shuffled)\n        _write(\n            out_dir / f"audit_annotator_{name}.csv",\n            [[item, s.text, *[""] * len(ASPECTS)] for item, s in shuffled],\n            ["item", "text", *ASPECTS],\n        )\n    _write(\n        out_dir / "audit_key.csv",\n        [[item, s.uid, "stratified" if i < stratified else "uniform_fill",\n          *[POLARITY_TO_CODE.get(int(v), "") for v in s.sentiment]] for i, (item, s) in enumerate(items)],\n        ["item", "uid", "stratum", *ASPECTS],\n    )\n    guide = ["# Hướng dẫn gán nhãn đối chứng", "",\n             "Làm độc lập, không trao đổi và không xem nhãn có sẵn.",\n             "Với mỗi câu, ở cột khía cạnh được nhắc đến ghi `pos`, `neg` hoặc `neu`; để trống các cột còn lại.",\n             "Một câu có thể có nhiều khía cạnh. Mỗi câu phải có ít nhất một khía cạnh; nếu không rõ, dùng `noi_chung`.", "",\n             "## 11 khía cạnh", ""]\n    guide += [f"- `{name}`: {ASPECT_DESCRIPTIONS[name]}" for name in ASPECTS]\n    (out_dir / "audit_guideline.md").write_text("\\n".join(guide) + "\\n", encoding="utf-8")\n\n\ndef cohen_kappa(a: np.ndarray, b: np.ndarray) -> float | None:\n    """Cohen\'s kappa for two label sequences. None if chance agreement is 1 (kappa undefined)."""\n    labels = np.unique(np.concatenate([a, b]))\n    observed = float((a == b).mean())\n    expected = sum(float((a == x).mean()) * float((b == x).mean()) for x in labels)\n    return None if expected >= 1.0 else (observed - expected) / (1.0 - expected)\n\n\ndef read_sheet(path: Path) -> dict[str, np.ndarray]:\n    """item -> polarity id per aspect, IGNORE_INDEX where blank. Invalid or unfinished sheets raise."""\n    table = {}\n    with path.open(encoding="utf-8-sig", newline="") as f:\n        for row in csv.DictReader(f):\n            values = []\n            for name in ASPECTS:\n                cell = row[name].strip().lower()\n                if cell and cell not in CODE_TO_POLARITY:\n                    raise ValueError(f"{path.name} {row[\'item\']}: invalid value {cell!r} in {name}")\n                values.append(CODE_TO_POLARITY.get(cell, IGNORE_INDEX))\n            if all(v == IGNORE_INDEX for v in values):\n                raise ValueError(f"{path.name} {row[\'item\']}: no aspect filled in")\n            table[row["item"]] = np.array(values)\n    return table\n\n\ndef agreement(a: np.ndarray, b: np.ndarray) -> dict:\n    """Agreement between two label matrices of shape (N, A)."""\n    per_aspect, both = {}, []\n    for i, name in enumerate(ASPECTS):\n        present_a, present_b = a[:, i] != IGNORE_INDEX, b[:, i] != IGNORE_INDEX\n        per_aspect[name] = {"kappa_presence": cohen_kappa(present_a, present_b), "n_a": int(present_a.sum()), "n_b": int(present_b.sum())}\n        shared = present_a & present_b\n        both.append((a[shared, i], b[shared, i]))\n    defined = [v["kappa_presence"] for v in per_aspect.values() if v["kappa_presence"] is not None]\n    sent_a, sent_b = np.concatenate([x for x, _ in both]), np.concatenate([y for _, y in both])\n    return {\n        "n_sentences": len(a),\n        "mean_kappa_presence": float(np.mean(defined)) if defined else None,\n        "kappa_sentiment_on_shared_aspects": cohen_kappa(sent_a, sent_b) if sent_a.size else None,\n        "n_shared_pairs": int(sent_a.size),\n        "exact_match_sentence": float((a == b).all(axis=1).mean()),\n        "per_aspect": per_aspect,\n    }\n\n\ndef score(sheet_dir: Path) -> dict:\n    a = read_sheet(sheet_dir / "audit_annotator_A.csv")\n    b = read_sheet(sheet_dir / "audit_annotator_B.csv")\n    current = {}\n    with (sheet_dir / "audit_key.csv").open(encoding="utf-8-sig", newline="") as f:\n        for row in csv.DictReader(f):\n            current[row["item"]] = np.array([CODE_TO_POLARITY.get(row[n], IGNORE_INDEX) for n in ASPECTS])\n    items = sorted(current)\n    if set(a) != set(items) or set(b) != set(items):\n        raise ValueError("sheets and key do not hold the same items")\n    A, B, C = (np.stack([t[i] for i in items]) for t in (a, b, current))\n    return {\n        "sampling": "stratified with rare aspects oversampled; not an unweighted estimate of the whole dataset",\n        "annotator_A_vs_B": agreement(A, B),\n        "annotator_A_vs_dataset": agreement(A, C),\n        "annotator_B_vs_dataset": agreement(B, C),\n    }\n\n\ndef main(argv=None) -> None:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("command", choices=["sample", "score"])\n    parser.add_argument("--data-dir", default="data/Final_Data")\n    parser.add_argument("--out-dir", default="data/annotation/audit")\n    parser.add_argument("--review-file", default="data/annotation/added_51_review.csv")\n    args = parser.parse_args(argv)\n    out_dir = Path(args.out_dir)\n\n    if args.command == "sample":\n        # The 51 added sentences did not go through the original workflow this audit measures.\n        with open(args.review_file, encoding="utf-8-sig", newline="") as f:\n            exclude = {f"test:{r[\'new_id\']}" for r in csv.DictReader(f) if r["original_split"] == "test"}\n        chosen, stratified = draw_sample(load_split(args.data_dir, "test"), exclude=exclude)\n        write_sheets(chosen, out_dir, stratified)\n        counts = {name: sum(int(s.aspect[i] > 0) for s in chosen) for i, name in enumerate(ASPECTS)}\n        print(json.dumps({"sentences": len(chosen), "stratified": stratified, "excluded_added": len(exclude), "per_aspect": counts}, indent=2))\n    else:\n        report = score(out_dir)\n        (out_dir / "audit_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n        print(json.dumps({k: v for k, v in report["annotator_A_vs_B"].items() if k != "per_aspect"}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'src/absa/baseline_tfidf.py': '"""TF-IDF + Linear SVM baseline (E0). Runs on CPU and writes the same files as `absa.train`.\n\n    python -m absa.baseline_tfidf --seed 42\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.svm import LinearSVC\n\nfrom .data import IGNORE_INDEX, NUM_ASPECTS, dataset_hash, label_arrays, load_split\nfrom .preprocess import normalize\nfrom .train import code_version\n\n\ndef fit_predict(train_texts, train_aspect, train_sentiment, eval_texts: dict, seed: int) -> dict:\n    """Train one detector and one sentiment classifier per aspect; predict each split in `eval_texts`.\n\n    The sentiment classifier for an aspect sees only sentences where that aspect exists.\n    Decision scores go through a sigmoid so the shared threshold grid applies.\n    """\n    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=True, min_df=2)\n    x_train = vectorizer.fit_transform(train_texts)\n    x_eval = {split: vectorizer.transform(texts) for split, texts in eval_texts.items()}\n    probs = {split: np.zeros((x.shape[0], NUM_ASPECTS), dtype=np.float32) for split, x in x_eval.items()}\n    sentiments = {split: np.zeros((x.shape[0], NUM_ASPECTS), dtype=np.int8) for split, x in x_eval.items()}\n\n    for a in range(NUM_ASPECTS):\n        detector = LinearSVC(C=1.0, class_weight="balanced", random_state=seed)\n        detector.fit(x_train, train_aspect[:, a].astype(int))\n        present = train_sentiment[:, a] != IGNORE_INDEX\n        labels = train_sentiment[present, a]\n        classifier = None\n        if len(set(labels.tolist())) > 1:\n            classifier = LinearSVC(C=1.0, class_weight="balanced", random_state=seed).fit(x_train[present], labels)\n        for split, x in x_eval.items():\n            probs[split][:, a] = 1.0 / (1.0 + np.exp(-detector.decision_function(x)))\n            sentiments[split][:, a] = classifier.predict(x) if classifier else (labels[0] if labels.size else 0)\n    return {"probs": probs, "sentiments": sentiments}\n\n\ndef run(args) -> Path:\n    started = time.time()\n    samples = {split: load_split(args.data_dir, split) for split in ("train", "validation", "test")}\n    texts = {split: [normalize(s.text) for s in rows] for split, rows in samples.items()}\n    aspect, sentiment = label_arrays(samples["train"])\n    output = fit_predict(texts["train"], aspect, sentiment, {k: texts[k] for k in ("validation", "test")}, args.seed)\n\n    run_name = f"e0_tfidf-svm_seed{args.seed}"\n    out_dir = Path(args.output_dir) / run_name\n    out_dir.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(\n        out_dir / "predictions.npz",\n        validation_uid=np.array([s.uid for s in samples["validation"]]),\n        validation_aspect_prob=output["probs"]["validation"],\n        validation_sentiment=output["sentiments"]["validation"],\n        test_uid=np.array([s.uid for s in samples["test"]]),\n        test_aspect_prob=output["probs"]["test"],\n        test_sentiment=output["sentiments"]["test"],\n    )\n    result = {\n        "run_name": run_name,\n        "config": {\n            "model": "e0", "seed": args.seed, "data_dir": args.data_dir, "limit": None,\n            "ngram_range": [1, 2], "min_df": 2, "C": 1.0, "class_weight": "balanced", "preprocess": "normalize+lower",\n        },\n        "code_commit": code_version(),\n        "dataset_hash": dataset_hash(args.data_dir),\n        "num_sentences": {split: len(rows) for split, rows in samples.items()},\n        "train_seconds": time.time() - started,\n        "versions": {"python": platform.python_version()},\n    }\n    (out_dir / "result.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(f"saved {out_dir}")\n    return out_dir\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--data-dir", default="data/Final_Data")\n    parser.add_argument("--output-dir", default="results/runs")\n    run(parser.parse_args())\n', 'src/absa/data.py': '"""Load `data/Final_Data` into one multi-label sample per sentence.\n\nIds are unique only within a file: the train files each restart from 1, so rows\nmust be grouped per file before the files are concatenated.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\n# Same order as the existing notebooks, so label indices stay comparable.\nASPECTS = (\n    "ky_nang_giang_day",\n    "hanh_vi",\n    "de_xuat",\n    "bai_tap",\n    "chuong_trinh_hoc",\n    "kien_thuc",\n    "kinh_nghiem",\n    "cung_cap_tai_lieu",\n    "thiet_bi_day_hoc",\n    "cham_diem",\n    "noi_chung",\n)\nPOLARITIES = ("negative", "neutral", "positive")\nNUM_ASPECTS = len(ASPECTS)\nNUM_POLARITIES = len(POLARITIES)\nASPECT2ID = {name: i for i, name in enumerate(ASPECTS)}\nPOLARITY2ID = {name: i for i, name in enumerate(POLARITIES)}\nIGNORE_INDEX = -100\n\nSPLIT_FILES = {\n    "train": tuple(f"train{i}.csv" for i in range(1, 7)),\n    "validation": ("validation.csv",),\n    "test": ("test.csv",),\n}\n\n\n@dataclass(frozen=True)\nclass Sample:\n    """One sentence. `uid` is unique across the whole dataset, `id` is not."""\n\n    uid: str\n    text: str\n    aspect: np.ndarray  # (NUM_ASPECTS,) float32 multi-hot\n    sentiment: np.ndarray  # (NUM_ASPECTS,) int64, IGNORE_INDEX where the aspect is absent\n\n\ndef load_file(path: str | Path) -> list[Sample]:\n    """Group one CSV by `id`. Bad rows raise instead of being skipped silently."""\n    path = Path(path)\n    groups: dict[int, list[dict[str, str]]] = {}\n    with path.open(encoding="utf-8", newline="") as f:\n        for row in csv.DictReader(f):\n            groups.setdefault(int(row["id"]), []).append(row)\n\n    samples = []\n    for sent_id in sorted(groups):\n        rows = groups[sent_id]\n        where = f"{path.name} id={sent_id}"\n        if len({r["text"] for r in rows}) != 1:\n            raise ValueError(f"{where}: one id holds different sentences")\n\n        aspect = np.zeros(NUM_ASPECTS, dtype=np.float32)\n        sentiment = np.full(NUM_ASPECTS, IGNORE_INDEX, dtype=np.int64)\n        for row in rows:\n            name, polarity = row["aspect"].strip(), row["sentiment"].strip()\n            if name not in ASPECT2ID:\n                raise ValueError(f"{where}: unknown aspect {name!r}")\n            if polarity not in POLARITY2ID:\n                raise ValueError(f"{where}: unknown sentiment {polarity!r}")\n            a, p = ASPECT2ID[name], POLARITY2ID[polarity]\n            if sentiment[a] not in (IGNORE_INDEX, p):\n                raise ValueError(f"{where}: conflicting sentiment for {name}")\n            aspect[a], sentiment[a] = 1.0, p\n\n        samples.append(Sample(f"{path.stem}:{sent_id}", rows[0]["text"], aspect, sentiment))\n    return samples\n\n\ndef load_split(data_dir: str | Path, split: str) -> list[Sample]:\n    """Load a split, grouping each file on its own before concatenating."""\n    samples: list[Sample] = []\n    for name in SPLIT_FILES[split]:\n        samples.extend(load_file(Path(data_dir) / name))\n    return samples\n\n\ndef label_arrays(samples: list[Sample]) -> tuple[np.ndarray, np.ndarray]:\n    """Stack samples into aspect (N, A) and sentiment (N, A) arrays."""\n    return np.stack([s.aspect for s in samples]), np.stack([s.sentiment for s in samples])\n\n\ndef dataset_hash(data_dir: str | Path) -> str:\n    """SHA-256 over all split files, to record which data a run actually used."""\n    digest = hashlib.sha256()\n    for split in ("train", "validation", "test"):\n        for name in SPLIT_FILES[split]:\n            digest.update(name.encode())\n            digest.update((Path(data_dir) / name).read_bytes())\n    return digest.hexdigest()\n', 'src/absa/evaluate.py': '"""Score a finished run. Thresholds always come from validation.\n\n    python -m absa.evaluate results/runs/e4_phobert-base_seed42\n    python -m absa.evaluate results/runs/e4_phobert-base_seed42 --test   # only after locking\n\nCompare two models across seeds (requires --test scoring to be allowed):\n\n    python -m absa.evaluate --compare results/runs/e4_*_seed4? --against results/runs/e2_*_seed4?\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .data import label_arrays, load_split\nfrom .metrics import evaluate, pair_counts, paired_bootstrap, predict_aspects, select_global_threshold, select_per_aspect_thresholds\n\n\ndef _labels(data_dir, split, uids, limit):\n    samples = load_split(data_dir, split)\n    if limit:\n        samples = samples[:limit]\n    if [s.uid for s in samples] != uids.tolist():\n        raise ValueError(f"{split}: predictions do not match the sentences in {data_dir}")\n    return label_arrays(samples)\n\n\ndef _load(run_dir: Path, data_dir: str | None):\n    result = json.loads((run_dir / "result.json").read_text(encoding="utf-8"))\n    return result, np.load(run_dir / "predictions.npz"), data_dir or result["config"]["data_dir"]\n\n\ndef _thresholds(result, predictions, data_dir):\n    limit = result["config"]["limit"]\n    gold = _labels(data_dir, "validation", predictions["validation_uid"], limit)\n    prob, sentiment = predictions["validation_aspect_prob"], predictions["validation_sentiment"]\n    return gold, select_global_threshold(*gold, prob, sentiment), select_per_aspect_thresholds(*gold, prob, sentiment)\n\n\ndef evaluate_run(run_dir: str | Path, data_dir: str | None = None, with_test: bool = False) -> dict:\n    run_dir = Path(run_dir)\n    result, predictions, data_dir = _load(run_dir, data_dir)\n    gold, global_t, per_aspect_t = _thresholds(result, predictions, data_dir)\n    decoders = {"fixed_0.5": 0.5, "global": global_t, "per_aspect": per_aspect_t}\n    prob, sentiment = predictions["validation_aspect_prob"], predictions["validation_sentiment"]\n\n    report = {\n        "run_name": result["run_name"],\n        "thresholds": {"global": global_t, "per_aspect": per_aspect_t.tolist()},\n        "validation": {name: evaluate(*gold, prob, sentiment, t) for name, t in decoders.items()},\n    }\n    if with_test:\n        test_gold = _labels(data_dir, "test", predictions["test_uid"], result["config"]["limit"])\n        report["test"] = {\n            name: evaluate(*test_gold, predictions["test_aspect_prob"], predictions["test_sentiment"], t)\n            for name, t in decoders.items()\n        }\n    name = "evaluation_test.json" if with_test else "evaluation_validation.json"\n    (run_dir / name).write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n    return report\n\n\ndef _sentence_counts(run_dir: Path, data_dir: str | None) -> np.ndarray:\n    """Per-sentence test TP, FP, FN at the run\'s validation-selected global threshold."""\n    result, predictions, data_dir = _load(run_dir, data_dir)\n    _, global_t, _ = _thresholds(result, predictions, data_dir)\n    test_gold = _labels(data_dir, "test", predictions["test_uid"], result["config"]["limit"])\n    pred_aspect = predict_aspects(predictions["test_aspect_prob"], global_t)\n    return pair_counts(*test_gold, pred_aspect, predictions["test_sentiment"]).sum(axis=1)\n\n\ndef compare(runs_a: list[str], runs_b: list[str], data_dir: str | None = None, n_boot: int = 1000) -> dict:\n    """Paired bootstrap of test `pair_f1_micro`. Both models must have run the same seeds."""\n    a, b = sorted(map(Path, runs_a)), sorted(map(Path, runs_b))\n    seeds_a = [json.loads((r / "result.json").read_text(encoding="utf-8"))["config"]["seed"] for r in a]\n    seeds_b = [json.loads((r / "result.json").read_text(encoding="utf-8"))["config"]["seed"] for r in b]\n    if seeds_a != seeds_b:\n        raise ValueError(f"models must use the same seeds: {seeds_a} vs {seeds_b}")\n    counts_a = np.stack([_sentence_counts(r, data_dir) for r in a])\n    counts_b = np.stack([_sentence_counts(r, data_dir) for r in b])\n    return {"seeds": seeds_a, **paired_bootstrap(counts_a, counts_b, n_boot=n_boot)}\n\n\ndef _print_summary(report: dict) -> None:\n    for split in ("validation", "test"):\n        for name, metrics in report.get(split, {}).items():\n            print(\n                f"{split:<10} {name:<10} aspect_f1={metrics[\'aspect_f1_micro\']:.4f} "\n                f"pair_f1={metrics[\'pair_f1_micro\']:.4f} pair_macro={metrics[\'pair_f1_macro\']:.4f} "\n                f"sent_acc_gold={metrics[\'sentiment_acc_on_gold\']:.4f}"\n            )\n    print("global threshold:", report["thresholds"]["global"])\n\n\ndef main(argv=None) -> None:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("run_dir", nargs="?")\n    parser.add_argument("--test", action="store_true", help="score the test set; only after the configuration is locked")\n    parser.add_argument("--compare", nargs="+", help="run directories of model A")\n    parser.add_argument("--against", nargs="+", help="run directories of model B")\n    parser.add_argument("--data-dir", default=None)\n    args = parser.parse_args(argv)\n    if args.compare:\n        print(json.dumps(compare(args.compare, args.against, args.data_dir), indent=2))\n    else:\n        _print_summary(evaluate_run(args.run_dir, args.data_dir, args.test))\n\n\nif __name__ == "__main__":\n    main()\n', 'src/absa/metrics.py': '"""Evaluation metrics, threshold selection, and paired bootstrap (docs/04-experiment-protocol.md).\n\nTask A is aspect detection. Task B counts a prediction as correct only if both the\naspect and its sentiment are right, so a wrong sentiment on a correctly detected\naspect is one false positive and one false negative.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .data import ASPECTS, IGNORE_INDEX, NUM_POLARITIES\n\n# Rounded so that values compare and serialize cleanly.\nTHRESHOLD_GRID = tuple(round(0.20 + 0.05 * i, 2) for i in range(11))  # 0.20 .. 0.70\n\n\ndef _prf(tp, fp, fn) -> tuple[float, float, float]:\n    """Precision, recall and F1 as fractions. An empty denominator gives 0."""\n    p = tp / (tp + fp) if tp + fp else 0.0\n    r = tp / (tp + fn) if tp + fn else 0.0\n    return p, r, (2 * p * r / (p + r) if p + r else 0.0)\n\n\ndef predict_aspects(aspect_prob: np.ndarray, threshold) -> np.ndarray:\n    """Binary aspect predictions. `threshold` is one value or one value per aspect."""\n    return aspect_prob > np.asarray(threshold)\n\n\ndef pair_counts(gold_aspect, gold_sentiment, pred_aspect, pred_sentiment) -> np.ndarray:\n    """Task B TP, FP, FN per sentence and aspect, shape (N, A, 3)."""\n    gold = gold_aspect.astype(bool)\n    pred = pred_aspect.astype(bool)\n    correct = gold & pred & (gold_sentiment == pred_sentiment)\n    return np.stack([correct, pred & ~correct, gold & ~correct], axis=-1).astype(np.int64)\n\n\ndef aspect_counts(gold_aspect, pred_aspect) -> np.ndarray:\n    """Task A TP, FP, FN per sentence and aspect, shape (N, A, 3)."""\n    gold = gold_aspect.astype(bool)\n    pred = pred_aspect.astype(bool)\n    return np.stack([gold & pred, pred & ~gold, gold & ~pred], axis=-1).astype(np.int64)\n\n\ndef _summarize(counts: np.ndarray, prefix: str) -> tuple[dict, list[tuple[float, float, float]]]:\n    per_aspect = [_prf(*counts[:, a].sum(axis=0)) for a in range(counts.shape[1])]\n    p, r, f1 = _prf(*counts.sum(axis=(0, 1)))\n    summary = {\n        f"{prefix}_p_micro": p,\n        f"{prefix}_r_micro": r,\n        f"{prefix}_f1_micro": f1,\n        f"{prefix}_f1_macro": float(np.mean([x[2] for x in per_aspect])),\n    }\n    return summary, per_aspect\n\n\ndef evaluate(gold_aspect, gold_sentiment, aspect_prob, pred_sentiment, threshold=0.5) -> dict:\n    """All protocol metrics for one model at one threshold (scalar or per aspect).\n\n    Args:\n        gold_aspect: (N, A) multi-hot gold aspects.\n        gold_sentiment: (N, A) gold polarity ids, IGNORE_INDEX where absent.\n        aspect_prob: (N, A) predicted aspect probabilities.\n        pred_sentiment: (N, A) argmax polarity id for every aspect.\n    """\n    pred_aspect = predict_aspects(aspect_prob, threshold)\n    aspect_summary, aspect_per = _summarize(aspect_counts(gold_aspect, pred_aspect), "aspect")\n    pair_summary, pair_per = _summarize(\n        pair_counts(gold_aspect, gold_sentiment, pred_aspect, pred_sentiment), "pair"\n    )\n\n    # Sentiment alone, on the aspects that truly exist. Independent of detection errors.\n    present = gold_sentiment != IGNORE_INDEX\n    gold_s, pred_s = gold_sentiment[present], pred_sentiment[present]\n    class_f1 = [\n        _prf(((gold_s == c) & (pred_s == c)).sum(), ((gold_s != c) & (pred_s == c)).sum(),\n             ((gold_s == c) & (pred_s != c)).sum())[2]\n        for c in range(NUM_POLARITIES)\n    ]\n    support = gold_aspect.astype(bool).sum(axis=0)\n    return {\n        **aspect_summary,\n        **pair_summary,\n        "sentiment_acc_on_gold": float((gold_s == pred_s).mean()) if gold_s.size else 0.0,\n        "sentiment_f1_macro_on_gold": float(np.mean(class_f1)),\n        "per_aspect": {\n            name: {\n                "support": int(support[a]),\n                "aspect_p": aspect_per[a][0],\n                "aspect_r": aspect_per[a][1],\n                "aspect_f1": aspect_per[a][2],\n                "pair_p": pair_per[a][0],\n                "pair_r": pair_per[a][1],\n                "pair_f1": pair_per[a][2],\n            }\n            for a, name in enumerate(ASPECTS)\n        },\n    }\n\n\ndef select_global_threshold(gold_aspect, gold_sentiment, aspect_prob, pred_sentiment) -> float:\n    """Grid value with the best `pair_f1_micro`; ties go to the lowest. Call on validation only."""\n    scores = [\n        _prf(*pair_counts(gold_aspect, gold_sentiment, predict_aspects(aspect_prob, t), pred_sentiment).sum(axis=(0, 1)))[2]\n        for t in THRESHOLD_GRID\n    ]\n    return THRESHOLD_GRID[int(np.argmax(scores))]\n\n\ndef select_per_aspect_thresholds(gold_aspect, gold_sentiment, aspect_prob, pred_sentiment) -> np.ndarray:\n    """Best grid value for each aspect\'s pair F1; ties go to the lowest. Call on validation only."""\n    best = np.zeros(gold_aspect.shape[1])\n    for a in range(gold_aspect.shape[1]):\n        scores = [\n            _prf(*pair_counts(gold_aspect[:, [a]], gold_sentiment[:, [a]],\n                              aspect_prob[:, [a]] > t, pred_sentiment[:, [a]]).sum(axis=(0, 1)))[2]\n            for t in THRESHOLD_GRID\n        ]\n        best[a] = THRESHOLD_GRID[int(np.argmax(scores))]\n    return best\n\n\ndef _micro_f1(tp: np.ndarray, fp: np.ndarray, fn: np.ndarray) -> np.ndarray:\n    denominator = 2 * tp + fp + fn\n    return np.divide(2 * tp, denominator, out=np.zeros_like(denominator, dtype=float), where=denominator > 0)\n\n\ndef paired_bootstrap(counts_a: np.ndarray, counts_b: np.ndarray, n_boot: int = 1000, seed: int = 0) -> dict:\n    """95% CI of the micro-F1 difference A - B, averaged over seeds.\n\n    Each resample draws the same sentence indices for both models and recomputes F1\n    from counts. The interval is conditional on the seeds that were run.\n\n    Args:\n        counts_a: Per-sentence (TP, FP, FN) of model A, shape (seeds, N, 3).\n        counts_b: The same for model B, with seeds in the same order.\n    """\n    if counts_a.shape != counts_b.shape:\n        raise ValueError("both models must have the same seeds and sentences")\n\n    def f1_gap(a: np.ndarray, b: np.ndarray) -> float:\n        total_a, total_b = a.sum(axis=1), b.sum(axis=1)  # (seeds, 3)\n        return float((_micro_f1(*total_a.T) - _micro_f1(*total_b.T)).mean())\n\n    n = counts_a.shape[1]\n    rng = np.random.default_rng(seed)\n    gaps = np.empty(n_boot)\n    for i in range(n_boot):\n        idx = rng.integers(0, n, size=n)\n        gaps[i] = f1_gap(counts_a[:, idx], counts_b[:, idx])\n    low, high = np.percentile(gaps, [2.5, 97.5])\n    return {"diff": f1_gap(counts_a, counts_b), "ci_low": float(low), "ci_high": float(high), "n_boot": n_boot}\n', 'src/absa/models/__init__.py': '"""Models. Both return (loss, aspect_prob, sentiment_logits) from `forward`."""\n', 'src/absa/models/bilstm_cnn.py': '"""BiLSTM-CNN baseline (E1), ported from `billstm_cnn.ipynb`.\n\nEach aspect has a 4-way output: none, positive, neutral, negative. An aspect counts\nas present when the argmax is not `none`, so no threshold applies: `aspect_prob` is\nreturned as 0 or 1 to fit the shared evaluation code.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom ..data import IGNORE_INDEX, NUM_ASPECTS, POLARITIES\n\nCLASS_NAMES = ("none", "positive", "neutral", "negative")\nNONE = 0\n# Positions in CLASS_NAMES, listed in POLARITIES order: converts 4-way logits to polarity logits.\nPOLARITY_CLASSES = [CLASS_NAMES.index(p) for p in POLARITIES]\n\n\ndef to_class_labels(sentiment: torch.Tensor) -> torch.Tensor:\n    """Polarity ids with IGNORE_INDEX -> 4-way labels with none = 0."""\n    labels = torch.zeros_like(sentiment)\n    for polarity_id, class_id in enumerate(POLARITY_CLASSES):\n        labels[sentiment == polarity_id] = class_id\n    return labels\n\n\ndef build_vocab(token_lists: list[list[str]]) -> dict[str, int]:\n    """Vocabulary from the training tokens only."""\n    vocab = {"<PAD>": 0, "<UNK>": 1}\n    for token in Counter(t for tokens in token_lists for t in tokens):\n        vocab[token] = len(vocab)\n    return vocab\n\n\ndef class_weights(sentiment: np.ndarray) -> torch.Tensor:\n    """Per-aspect square-root inverse frequency, shape (A, 4). Unseen classes get 1."""\n    counts = np.zeros((NUM_ASPECTS, len(CLASS_NAMES)))\n    labels = to_class_labels(torch.as_tensor(sentiment)).numpy()\n    for a in range(NUM_ASPECTS):\n        for c in range(len(CLASS_NAMES)):\n            counts[a, c] = (labels[:, a] == c).sum()\n    total = counts.sum(axis=1, keepdims=True)\n    with np.errstate(divide="ignore"):\n        weights = np.sqrt(total / (len(CLASS_NAMES) * counts))\n    weights[counts == 0] = 1.0\n    return torch.tensor(weights, dtype=torch.float)\n\n\nclass BiLSTMCNN(nn.Module):\n    def __init__(\n        self,\n        vocab_size: int,\n        weights: torch.Tensor,\n        embedding_dim: int = 300,\n        hidden_size: int = 128,\n        num_filters: int = 128,\n        kernel_sizes: tuple[int, ...] = (2, 3, 4),\n        fc_hidden: int = 128,\n        dropout: float = 0.6,\n        pretrained_embeddings: np.ndarray | None = None,\n    ):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)\n        if pretrained_embeddings is not None:\n            self.embedding.weight.data.copy_(torch.as_tensor(pretrained_embeddings, dtype=torch.float))\n            self.embedding.weight.data[0] = 0.0\n        self.bilstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True, bidirectional=True)\n        self.convs = nn.ModuleList(nn.Conv1d(hidden_size * 2, num_filters, k) for k in kernel_sizes)\n        self.dropout = nn.Dropout(dropout)\n        self.fc = nn.Linear(num_filters * len(kernel_sizes), fc_hidden)\n        self.output = nn.Linear(fc_hidden, NUM_ASPECTS * len(CLASS_NAMES))\n        self.register_buffer("class_weights", weights)\n\n    def forward(self, input_ids, attention_mask=None, aspect_labels=None, sentiment_labels=None):\n        lstm_out, _ = self.bilstm(self.embedding(input_ids))\n        x = lstm_out.transpose(1, 2)\n        pooled = [F.relu(conv(x)).max(dim=2).values for conv in self.convs]\n        x = self.dropout(torch.cat(pooled, dim=1))\n        x = self.dropout(F.relu(self.fc(x)))\n        logits = self.output(x).view(-1, NUM_ASPECTS, len(CLASS_NAMES))\n\n        loss = None\n        if sentiment_labels is not None:\n            labels = to_class_labels(sentiment_labels)\n            loss = sum(\n                F.cross_entropy(logits[:, a], labels[:, a], weight=self.class_weights[a]) for a in range(NUM_ASPECTS)\n            ) / NUM_ASPECTS\n\n        aspect_prob = (logits.argmax(dim=-1) != NONE).float()\n        return loss, aspect_prob, logits[:, :, POLARITY_CLASSES]\n\n\ndef load_word2vec(vocab: dict[str, int], path: str, embedding_dim: int = 300, seed: int = 0) -> tuple[np.ndarray, float]:\n    """Word2Vec matrix for the vocabulary; compounds fall back to averaging their syllables."""\n    from gensim.models import KeyedVectors  # only this baseline needs gensim\n\n    vectors = KeyedVectors.load_word2vec_format(path, binary=path.endswith(".bin"))\n    if vectors.vector_size != embedding_dim:\n        raise ValueError(f"expected {embedding_dim}-dim vectors, found {vectors.vector_size}")\n    rng = np.random.default_rng(seed)\n    matrix = rng.uniform(-0.1, 0.1, (len(vocab), embedding_dim)).astype(np.float32)\n    matrix[0] = 0.0\n    found = 0\n    for word, index in vocab.items():\n        if index < 2:\n            continue\n        if word in vectors:\n            matrix[index] = vectors[word]\n            found += 1\n        else:\n            parts = [vectors[p] for p in word.split("_") if p in vectors]\n            if parts and "_" in word:\n                matrix[index] = np.mean(parts, axis=0)\n                found += 1\n    return matrix, found / max(len(vocab) - 2, 1)\n\n\n__all__ = ["BiLSTMCNN", "IGNORE_INDEX", "build_vocab", "class_weights", "load_word2vec", "to_class_labels"]\n', 'src/absa/models/multitask.py': '"""Multi-task ABSA on a Transformer encoder: experiments E2, E3, E5 and E4.\n\n| Experiment | Aspect branch | Sentiment branch | Query init |\n|------------|---------------|------------------|------------|\n| E2         | mean pooling  | mean pooling     | -          |\n| E3         | mean pooling  | aspect query     | random     |\n| E5         | aspect query  | aspect query     | random     |\n| E4         | aspect query  | aspect query     | descriptions |\n\nE2 and E3 reproduce the two heads in `phobert_multitask.ipynb`.\n"""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\n\nfrom ..data import IGNORE_INDEX\n\nPRESETS = {\n    "e2": {"aspect_branch": "mean", "sentiment_branch": "mean", "query_init": None},\n    "e3": {"aspect_branch": "mean", "sentiment_branch": "query", "query_init": "random"},\n    "e5": {"aspect_branch": "query", "sentiment_branch": "query", "query_init": "random"},\n    "e4": {"aspect_branch": "query", "sentiment_branch": "query", "query_init": "descriptions"},\n}\n\n\ndef mean_pooling(hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    """Average token states, ignoring padding."""\n    mask = attention_mask.unsqueeze(-1).float()\n    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)\n\n\nclass MultiTaskABSA(nn.Module):\n    def __init__(\n        self,\n        encoder: nn.Module,\n        num_aspects: int,\n        num_polarities: int,\n        pos_weight: torch.Tensor,\n        sentiment_class_weight: torch.Tensor,\n        aspect_branch: str = "mean",\n        sentiment_branch: str = "mean",\n        alpha: float = 1.0,\n        beta: float = 1.0,\n        dropout: float = 0.3,\n        num_heads: int = 8,\n    ):\n        super().__init__()\n        if aspect_branch not in ("mean", "query") or sentiment_branch not in ("mean", "query"):\n            raise ValueError("branches must be \'mean\' or \'query\'")\n        if aspect_branch == "query" and sentiment_branch != "query":\n            raise ValueError("no experiment uses query attention for aspects only")\n\n        hidden = encoder.config.hidden_size\n        self.encoder = encoder\n        self.num_aspects, self.num_polarities = num_aspects, num_polarities\n        self.aspect_branch, self.sentiment_branch = aspect_branch, sentiment_branch\n        self.alpha, self.beta = alpha, beta\n        self.dropout = nn.Dropout(dropout)\n        self.register_buffer("pos_weight", pos_weight)\n        self.register_buffer("sentiment_class_weight", sentiment_class_weight)\n\n        if sentiment_branch == "query":\n            self.aspect_embedding = nn.Embedding(num_aspects, hidden)\n            self.cross_attention = nn.MultiheadAttention(hidden, num_heads, dropout=dropout, batch_first=True)\n            self.sentiment_head = nn.Sequential(\n                nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, num_polarities)\n            )\n        else:\n            self.sentiment_head = nn.Linear(hidden, num_aspects * num_polarities)\n\n        if aspect_branch == "query":\n            # One linear classifier per aspect, each on that aspect\'s own vector.\n            self.aspect_weight = nn.Parameter(torch.empty(num_aspects, hidden))\n            self.aspect_bias = nn.Parameter(torch.zeros(num_aspects))\n            nn.init.xavier_uniform_(self.aspect_weight)\n        else:\n            self.aspect_head = nn.Linear(hidden, num_aspects)\n\n    def set_queries(self, queries: torch.Tensor) -> None:\n        """Replace the initial queries (E4). They keep training afterwards."""\n        self.aspect_embedding.weight.data.copy_(queries)\n\n    def forward(self, input_ids, attention_mask, aspect_labels=None, sentiment_labels=None):\n        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n        pooled = self.dropout(mean_pooling(hidden, attention_mask))\n\n        if self.sentiment_branch == "query":\n            query = self.aspect_embedding.weight.unsqueeze(0).expand(input_ids.size(0), -1, -1)\n            sequence = self.dropout(hidden)\n            attended, _ = self.cross_attention(\n                query, sequence, sequence, key_padding_mask=attention_mask == 0\n            )\n            sentiment_logits = self.sentiment_head(torch.cat([attended, query], dim=-1))\n        else:\n            sentiment_logits = self.sentiment_head(pooled).view(-1, self.num_aspects, self.num_polarities)\n\n        if self.aspect_branch == "query":\n            aspect_logits = (attended * self.aspect_weight).sum(dim=-1) + self.aspect_bias\n        else:\n            aspect_logits = self.aspect_head(pooled)\n\n        loss = None\n        if aspect_labels is not None:\n            aspect_loss = nn.functional.binary_cross_entropy_with_logits(\n                aspect_logits, aspect_labels, pos_weight=self.pos_weight\n            )\n            sentiment_loss = nn.functional.cross_entropy(\n                sentiment_logits.reshape(-1, self.num_polarities),\n                sentiment_labels.reshape(-1),\n                weight=self.sentiment_class_weight,\n                ignore_index=IGNORE_INDEX,\n            )\n            loss = self.alpha * aspect_loss + self.beta * sentiment_loss\n        return loss, torch.sigmoid(aspect_logits), sentiment_logits\n\n\n@torch.no_grad()\ndef encode_descriptions(encoder, tokenizer, texts: list[str], device, max_length: int = 128) -> torch.Tensor:\n    """Mean-pooled encoder vectors of the aspect descriptions, shape (A, H)."""\n    was_training = encoder.training\n    encoder.eval()\n    batch = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)\n    hidden = encoder(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state\n    vectors = mean_pooling(hidden, batch["attention_mask"])\n    encoder.train(was_training)\n    return vectors\n', 'src/absa/preprocess.py': '"""Text preprocessing. Each encoder declares which mode it needs."""\n\nfrom __future__ import annotations\n\nimport re\nimport unicodedata\n\n\ndef normalize(text: str) -> str:\n    """NFC-normalize, trim, and drop leading dots or dashes."""\n    text = unicodedata.normalize("NFC", str(text).strip())\n    return re.sub(r"^[\\s.\\-]+", "", text)\n\n\ndef preprocess(text: str, mode: str) -> str:\n    """`underthesea`: word-segmented, as in the PhoBERT notebook. `raw`: normalized only."""\n    text = normalize(text)\n    if mode == "raw":\n        return text\n    if mode == "underthesea":\n        from underthesea import word_tokenize  # slow to import; only PhoBERT needs it\n\n        return word_tokenize(text, format="text")\n    raise ValueError(f"unknown preprocessing mode: {mode!r}")\n', 'src/absa/train.py': '"""Train one experiment on one seed and save validation and test probabilities.\n\nTest probabilities are written but never scored here: `absa.evaluate --test` scores them\nafter the configuration is locked (docs/04-experiment-protocol.md).\n\n    python -m absa.train --model e4 --encoder vinai/phobert-base --seed 42\n    python -m absa.train --model bilstm_cnn --word2vec baomoi.vn.model.bin --seed 42\n\nTo reproduce the old notebooks (gate G2), add `--selection legacy` and, for E3, `--beta 1.5`.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport json\nimport os\nimport platform\nimport random\nimport subprocess\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .aspects import aspect_texts\nfrom .data import NUM_ASPECTS, NUM_POLARITIES, dataset_hash, label_arrays, load_split\nfrom .metrics import evaluate\nfrom .models.bilstm_cnn import BiLSTMCNN, build_vocab, class_weights, load_word2vec\nfrom .models.multitask import PRESETS, MultiTaskABSA, encode_descriptions\nfrom .preprocess import preprocess\n\n# Preprocessing is part of the model. Other encoders must pass --preprocess explicitly.\nENCODER_PREPROCESS = {"vinai/phobert-base": "underthesea", "Qualcomm-AI-Research/BamiBERT": "raw"}\nSPLITS = ("train", "validation", "test")\nMAX_LENGTH = 128\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef sqrt_pos_weight(aspect: np.ndarray) -> torch.Tensor:\n    """Square root of the negative/positive ratio, clipped to [1, 20], as in the notebook."""\n    positive = aspect.sum(axis=0)\n    ratio = (len(aspect) - positive) / (positive + 1e-8)\n    return torch.tensor(np.clip(np.sqrt(ratio), 1.0, 20.0), dtype=torch.float)\n\n\nclass EncodedDataset(Dataset):\n    def __init__(self, input_ids, aspect, sentiment):\n        self.input_ids, self.aspect, self.sentiment = input_ids, aspect, sentiment\n\n    def __len__(self):\n        return len(self.input_ids)\n\n    def __getitem__(self, i):\n        return self.input_ids[i], self.aspect[i], self.sentiment[i]\n\n\ndef collate(pad_id: int, pad_to: int | None = None):\n    """Pad to the longest in the batch, or to `pad_to` (the LSTM runs over padding, so length matters)."""\n\n    def fn(items):\n        ids, aspect, sentiment = zip(*items)\n        length = pad_to or max(len(x) for x in ids)\n        input_ids = torch.full((len(ids), length), pad_id, dtype=torch.long)\n        mask = torch.zeros((len(ids), length), dtype=torch.long)\n        for row, x in enumerate(ids):\n            input_ids[row, : len(x)] = torch.tensor(x, dtype=torch.long)\n            mask[row, : len(x)] = 1\n        return {\n            "input_ids": input_ids,\n            "attention_mask": mask,\n            "aspect_labels": torch.as_tensor(np.stack(aspect), dtype=torch.float),\n            "sentiment_labels": torch.as_tensor(np.stack(sentiment), dtype=torch.long),\n        }\n\n    return fn\n\n\n@torch.no_grad()\ndef predict(model, loader, device) -> tuple[np.ndarray, np.ndarray]:\n    model.eval()\n    probabilities, sentiments = [], []\n    for batch in loader:\n        _, aspect_prob, sentiment_logits = model(\n            batch["input_ids"].to(device), batch["attention_mask"].to(device)\n        )\n        probabilities.append(aspect_prob.cpu().numpy())\n        sentiments.append(sentiment_logits.argmax(dim=-1).cpu().numpy())\n    return np.concatenate(probabilities), np.concatenate(sentiments)\n\n\ndef score(metrics: dict, selection: str, model_name: str) -> float:\n    if selection == "pair":\n        return metrics["pair_f1_micro"]\n    if model_name == "bilstm_cnn":  # legacy: billstm_cnn.ipynb selected on aspect F1\n        return metrics["aspect_f1_micro"]\n    return (metrics["aspect_f1_micro"] * metrics["sentiment_f1_macro_on_gold"]) ** 0.5\n\n\ndef code_version() -> str | None:\n    """Commit of this code. Kaggle has no git history, so it reads ABSA_COMMIT there."""\n    if os.environ.get("ABSA_COMMIT"):\n        return os.environ["ABSA_COMMIT"]\n    try:\n        root = Path(__file__).resolve().parent\n        commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=root, capture_output=True, text=True, check=True).stdout.strip()\n        dirty = subprocess.run(["git", "status", "--porcelain", "--", "."], cwd=root, capture_output=True, text=True, check=True).stdout.strip()\n        return commit + ("+dirty" if dirty else "")\n    except (OSError, subprocess.CalledProcessError):\n        return None\n\n\ndef prepare_transformer(args, samples, device):\n    from transformers import AutoModel, AutoTokenizer\n\n    mode = args.preprocess or ENCODER_PREPROCESS.get(args.encoder)\n    if mode is None:\n        raise SystemExit(f"--preprocess is required for {args.encoder}; check its model card")\n    tokenizer = AutoTokenizer.from_pretrained(args.encoder, revision=args.revision)\n    encoder = AutoModel.from_pretrained(args.encoder, revision=args.revision)\n\n    encoded, truncation = {}, {}\n    for split in SPLITS:\n        texts = [preprocess(s.text, mode) for s in samples[split]]\n        ids = tokenizer(texts, truncation=False, add_special_tokens=True)["input_ids"]\n        truncation[split] = sum(len(x) > MAX_LENGTH for x in ids) / max(len(ids), 1)\n        encoded[split] = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)["input_ids"]\n\n    aspect, _ = label_arrays(samples["train"])\n    preset = PRESETS[args.model]\n    model = MultiTaskABSA(\n        encoder, NUM_ASPECTS, NUM_POLARITIES,\n        pos_weight=sqrt_pos_weight(aspect),\n        sentiment_class_weight=torch.tensor([1.0, 2.0, 1.0]),\n        aspect_branch=preset["aspect_branch"],\n        sentiment_branch=preset["sentiment_branch"],\n        alpha=args.alpha, beta=args.beta, dropout=0.3,\n    ).to(device)\n    if preset["query_init"] == "descriptions":\n        texts = [preprocess(t, mode) for t in aspect_texts()]\n        model.set_queries(encode_descriptions(encoder, tokenizer, texts, device, MAX_LENGTH))\n\n    no_decay = ("bias", "LayerNorm.weight")\n    optimizer = torch.optim.AdamW(\n        [\n            {"params": [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], "weight_decay": 0.01},\n            {"params": [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], "weight_decay": 0.0},\n        ],\n        lr=args.lr or 2e-5, eps=1e-8,\n    )\n    settings = {"batch_size": 16, "eval_batch_size": 32, "epochs": 8, "patience": 2, "lr": args.lr or 2e-5, "preprocess": mode}\n    extra = {"truncation_rate": truncation, "model_commit": getattr(encoder.config, "_commit_hash", None)}\n    return model, optimizer, encoded, collate(tokenizer.pad_token_id), settings, extra\n\n\ndef prepare_bilstm(args, samples, device):\n    tokens = {split: [preprocess(s.text, "underthesea").lower().split() for s in samples[split]] for split in SPLITS}\n    vocab = build_vocab(tokens["train"])\n    pretrained, coverage = None, None\n    if args.word2vec:\n        pretrained, coverage = load_word2vec(vocab, args.word2vec, seed=args.seed)\n    _, sentiment = label_arrays(samples["train"])\n    model = BiLSTMCNN(len(vocab), class_weights(sentiment), pretrained_embeddings=pretrained).to(device)\n    lr = args.lr or 5e-4\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)\n    encoded = {split: [[vocab.get(t, 1) for t in row][:MAX_LENGTH] for row in rows] for split, rows in tokens.items()}\n    truncation = {split: sum(len(r) > MAX_LENGTH for r in rows) / max(len(rows), 1) for split, rows in tokens.items()}\n    settings = {"batch_size": 32, "eval_batch_size": 64, "epochs": 30, "patience": 5, "lr": lr, "preprocess": "underthesea+lower"}\n    extra = {\n        "truncation_rate": truncation, "vocab_size": len(vocab),\n        "word2vec": args.word2vec, "word2vec_coverage": coverage,\n    }\n    return model, optimizer, encoded, collate(0, pad_to=MAX_LENGTH), settings, extra\n\n\ndef run(args) -> Path:\n    set_seed(args.seed)\n    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    samples = {split: load_split(args.data_dir, split) for split in SPLITS}\n    if args.limit:\n        samples = {split: rows[: args.limit] for split, rows in samples.items()}\n\n    is_bilstm = args.model == "bilstm_cnn"\n    prepare = prepare_bilstm if is_bilstm else prepare_transformer\n    model, optimizer, encoded, collate_fn, settings, extra = prepare(args, samples, device)\n    epochs = args.epochs or settings["epochs"]\n\n    def loader(split, shuffle=False, batch_size=None):\n        aspect, sentiment = label_arrays(samples[split])\n        return DataLoader(\n            EncodedDataset(encoded[split], aspect, sentiment),\n            batch_size=batch_size or settings["eval_batch_size"],\n            shuffle=shuffle,\n            collate_fn=collate_fn,\n            generator=torch.Generator().manual_seed(args.seed),\n        )\n\n    train_loader = loader("train", shuffle=True, batch_size=settings["batch_size"])\n    val_loader, test_loader = loader("validation"), loader("test")\n    val_aspect, val_sentiment = label_arrays(samples["validation"])\n\n    if is_bilstm:\n        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)\n    else:\n        from transformers import get_linear_schedule_with_warmup\n\n        total = len(train_loader) * epochs\n        scheduler = get_linear_schedule_with_warmup(optimizer, int(total * 0.1), total)\n\n    backbone = ("word2vec" if args.word2vec else "random-embeddings") if is_bilstm else args.encoder.split("/")[-1]\n    run_name = args.run_name or f"{args.model}_{backbone}_seed{args.seed}"\n    out_dir = Path(args.output_dir) / run_name\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n    started = time.time()\n    best_score, best_epoch, best_state, stale, history = -1.0, 0, None, 0, []\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        losses = []\n        for batch in train_loader:\n            batch = {k: v.to(device) for k, v in batch.items()}\n            loss, _, _ = model(**batch)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            if not is_bilstm:\n                scheduler.step()\n            optimizer.zero_grad()\n            losses.append(loss.item())\n\n        probs, sentiments = predict(model, val_loader, device)\n        metrics = evaluate(val_aspect, val_sentiment, probs, sentiments, 0.5)\n        current = score(metrics, args.selection, args.model)\n        if is_bilstm:\n            scheduler.step(current)\n        history.append({\n            "epoch": epoch, "train_loss": float(np.mean(losses)), "val_score": current,\n            "val_aspect_f1_micro": metrics["aspect_f1_micro"], "val_pair_f1_micro": metrics["pair_f1_micro"],\n            "seconds": time.time() - started,\n        })\n        print(json.dumps(history[-1]), flush=True)\n\n        if current > best_score:\n            best_score, best_epoch, stale = current, epoch, 0\n            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})\n        else:\n            stale += 1\n            if stale >= settings["patience"]:\n                break\n\n    model.load_state_dict(best_state)\n    model.to(device)\n    val_probs, val_sentiments = predict(model, val_loader, device)\n    test_probs, test_sentiments = predict(model, test_loader, device)\n    np.savez_compressed(\n        out_dir / "predictions.npz",\n        validation_uid=np.array([s.uid for s in samples["validation"]]),\n        validation_aspect_prob=val_probs.astype(np.float32),\n        validation_sentiment=val_sentiments.astype(np.int8),\n        test_uid=np.array([s.uid for s in samples["test"]]),\n        test_aspect_prob=test_probs.astype(np.float32),\n        test_sentiment=test_sentiments.astype(np.int8),\n    )\n    if args.save_checkpoint:\n        torch.save(best_state, out_dir / "best_model.pt")\n\n    result = {\n        "run_name": run_name,\n        "config": {**{k: v for k, v in vars(args).items()}, **settings, "epochs": epochs},\n        "code_commit": code_version(),\n        "dataset_hash": dataset_hash(args.data_dir),\n        "limited_smoke_run": bool(args.limit),\n        "num_sentences": {split: len(rows) for split, rows in samples.items()},\n        **extra,\n        "best_epoch": best_epoch,\n        "best_val_score": best_score,\n        "history": history,\n        "train_seconds": time.time() - started,\n        "peak_gpu_memory_mb": torch.cuda.max_memory_allocated(device) / 2**20 if device.type == "cuda" else None,\n        "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__},\n    }\n    (out_dir / "result.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(f"saved {out_dir}", flush=True)\n    return out_dir\n\n\ndef parse_args(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--model", required=True, choices=[*PRESETS, "bilstm_cnn"])\n    parser.add_argument("--seed", type=int, required=True)\n    parser.add_argument("--encoder", default="vinai/phobert-base")\n    parser.add_argument("--revision", default=None, help="pin the encoder revision")\n    parser.add_argument("--preprocess", default=None, choices=["underthesea", "raw"])\n    parser.add_argument("--word2vec", default=None, help="BiLSTM-CNN only; random embeddings if omitted")\n    parser.add_argument("--data-dir", default="data/Final_Data")\n    parser.add_argument("--output-dir", default="results/runs")\n    parser.add_argument("--run-name", default=None)\n    parser.add_argument("--alpha", type=float, default=1.0)\n    parser.add_argument("--beta", type=float, default=1.0, help="protocol value is 1.0; 1.5 only to reproduce E3")\n    parser.add_argument("--selection", default="pair", choices=["pair", "legacy"])\n    parser.add_argument("--lr", type=float, default=None)\n    parser.add_argument("--epochs", type=int, default=None)\n    parser.add_argument("--limit", type=int, default=None, help="smoke test: use only the first N sentences per split")\n    parser.add_argument("--device", default=None)\n    parser.add_argument("--save-checkpoint", action="store_true")\n    return parser.parse_args(argv)\n\n\nif __name__ == "__main__":\n    run(parse_args())\n'}
for name, text in FILES.items():
    path = Path(WORK) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
print(len(FILES), 'source files written')


In [ ]:
sys.path.insert(0, os.path.join(WORK, "src"))
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
os.environ["ABSA_COMMIT"] = 'e89f1ce+src-9c6a93d3e514'
from absa.data import dataset_hash

def find_one(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

data = os.path.dirname(find_one("/kaggle/input/**/train1.csv"))
word2vec = next(iter(glob.glob("/kaggle/input/**/baomoi.vn.model.bin", recursive=True)), "")
actual = dataset_hash(data)
print("data dir:", data)
print("dataset hash:", actual, "| expected:", 'a947ca4a9e5ed36625eb97abe9e63f2f4f5b66e66f109b31e665890cb3e05899', "| match:", actual == 'a947ca4a9e5ed36625eb97abe9e63f2f4f5b66e66f109b31e665890cb3e05899')

for template in ['python -m absa.train --model e3 --seed 42 --beta 1.5 --selection legacy --data-dir {data} --output-dir results/g2 --run-name e3_seed42_legacy', 'python -m absa.evaluate results/g2/e3_seed42_legacy --test', 'python -m absa.train --model bilstm_cnn --seed 42 --selection legacy --word2vec {word2vec} --data-dir {data} --output-dir results/g2 --run-name e1_seed42_legacy', 'python -m absa.evaluate results/g2/e1_seed42_legacy --test']:
    command = template.format(data=data, word2vec=word2vec)
    print("\n$", command, flush=True)
    started = time.time()
    process = subprocess.Popen(command.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"failed: {command}")
    print(f"[{time.time() - started:.0f} s]", flush=True)
